In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU not detected")

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [ ]:
!pip install -q transformers contractions emoji beautifulsoup4 nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.8 MB/s eta 0:00:00


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import os

MODEL_PATH = "/content/drive/MyDrive/Data_Science/models/DistilBERT_Model"

print("Model exists:", os.path.exists(MODEL_PATH))

if os.path.exists(MODEL_PATH):
    print("Files:")
    print(os.listdir(MODEL_PATH))

Model exists: True
Files:
['config.json', 'model.safetensors', 'training_args.bin', 'tokenizer_config.json', 'tokenizer.json']


In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

MODEL_PATH = "/content/drive/MyDrive/Data_Science/models/DistilBERT_Model"

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH
)

model.to(device)
model.eval()

print("Model loaded successfully!")
print("Device:", device)
print("Number of labels:", model.config.num_labels)
print("Label mapping:", model.config.id2label)

Loading weights:   0%|          | 0/104 [00:06<?, ?it/s]

Model loaded successfully!
Device: cuda
Number of labels: 6
Label mapping: {0: 'toxic', 1: 'severe_toxic', 2: 'obscene', 3: 'threat', 4: 'insult', 5: 'identity_hate'}


In [ ]:
import pandas as pd
import os

TEST_PATH = "/content/drive/MyDrive/Data_Science/models/test_cleaned.csv"

print("File exists:", os.path.exists(TEST_PATH))

if os.path.exists(TEST_PATH):
    test_df = pd.read_csv(TEST_PATH)

    print("Shape:", test_df.shape)
    print("\nColumns:")
    print(test_df.columns.tolist())

    display(test_df.head())

File exists: True
Shape: (153164, 3)

Columns:
['id', 'comment_text', 'clean_text']


,id,comment_text,clean_text
0,00001cee341fdb12,Yo bitch Ja Rule is more succesful then you'll...,yo bitch ja rule is more succesful then you wi...
1,0000247867823ef7,== From RfC == \n\n The title is fine as it is...,from rfc the title is fine a it is i am going to
2,00013b17ad220c46,""" \n\n == Sources == \n\n * Zawe Ashton on Lap...",source zawe ashton on lapland
3,00017563c3f7919a,":If you have a look back at the source, the in...",if you have a look back at the source the info...
4,00017695ad8997eb,I don't anonymously edit articles at all.,i do not anonymously edit article at all


In [ ]:
import numpy as np
from tqdm.auto import tqdm

LABELS = [
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate",
]

MAX_LENGTH = 150
THRESHOLD = 0.5
BATCH_SIZE = 64


def predict_binary_batch(texts, batch_size=64):

    all_predictions = []

    for start in tqdm(
        range(0, len(texts), batch_size),
        desc="Predicting"
    ):

        batch_texts = texts[
            start:start + batch_size
        ]

        inputs = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        )

        inputs = {
            key: value.to(device)
            for key, value in inputs.items()
        }

        with torch.inference_mode():

            outputs = model(**inputs)

            probabilities = torch.sigmoid(
                outputs.logits
            )

            predictions = (
                probabilities >= THRESHOLD
            ).int()

        all_predictions.append(
            predictions.cpu().numpy()
        )

    return np.vstack(all_predictions)

In [ ]:
sample_df = test_df.head(100).copy()

sample_texts = (
    sample_df["clean_text"]
    .fillna("")
    .astype(str)
    .tolist()
)

sample_predictions = predict_binary_batch(
    sample_texts,
    batch_size=BATCH_SIZE,
)

print("Prediction shape:", sample_predictions.shape)

print(
    "Unique prediction values:",
    np.unique(sample_predictions)
)

Predicting:   0%|          | 0/2 [00:00<?, ?it/s]

Prediction shape: (100, 6)
Unique prediction values: [0 1]


In [ ]:
for i, label in enumerate(LABELS):
    sample_df[label] = sample_predictions[:, i]

display(
    sample_df[
        [
            "id",
            "comment_text",
            "toxic",
            "severe_toxic",
            "obscene",
            "threat",
            "insult",
            "identity_hate",
        ]
    ].head(20)
)

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,00001cee341fdb12,Yo bitch Ja Rule is more succesful then you'll...,1,0,1,0,1,1
1,0000247867823ef7,== From RfC == \n\n The title is fine as it is...,0,0,0,0,0,0
2,00013b17ad220c46,""" \n\n == Sources == \n\n * Zawe Ashton on Lap...",0,0,0,0,0,0
3,00017563c3f7919a,":If you have a look back at the source, the in...",0,0,0,0,0,0
4,00017695ad8997eb,I don't anonymously edit articles at all.,0,0,0,0,0,0
5,0001ea8717f6de06,Thank you for understanding. I think very high...,0,0,0,0,0,0
6,00024115d4cbde0f,Please do not add nonsense to Wikipedia. Such ...,0,0,0,0,0,0
7,000247e83dcc1211,:Dear god this site is horrible.,1,0,0,0,0,0
8,00025358d4737918,""" \n Only a fool can believe in such numbers. ...",0,0,0,0,0,0
9,00026d1092fe71cc,== Double Redirects == \n\n When fixing double...,0,0,0,0,0,0


In [ ]:
# Prepare all preprocessed comments
all_texts = (
    test_df["clean_text"]
    .fillna("")
    .astype(str)
    .tolist()
)

print(f"Total comments to predict: {len(all_texts):,}")

# Run GPU batch inference
all_predictions = predict_binary_batch(
    all_texts,
    batch_size=64,
)

print("\nPrediction completed!")
print("Prediction shape:", all_predictions.shape)
print("Unique values:", np.unique(all_predictions))

Total comments to predict: 153,164


Predicting:   0%|          | 0/2394 [00:00<?, ?it/s]


Prediction completed!
Prediction shape: (153164, 6)
Unique values: [0 1]


In [ ]:
final_df = test_df[
    ["id", "comment_text"]
].copy()

for i, label in enumerate(LABELS):
    final_df[label] = all_predictions[:, i]

print("Final shape:", final_df.shape)

display(final_df.head(20))

Final shape: (153164, 8)


,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,00001cee341fdb12,Yo bitch Ja Rule is more succesful then you'll...,1,0,1,0,1,1
1,0000247867823ef7,== From RfC == \n\n The title is fine as it is...,0,0,0,0,0,0
2,00013b17ad220c46,""" \n\n == Sources == \n\n * Zawe Ashton on Lap...",0,0,0,0,0,0
3,00017563c3f7919a,":If you have a look back at the source, the in...",0,0,0,0,0,0
4,00017695ad8997eb,I don't anonymously edit articles at all.,0,0,0,0,0,0
5,0001ea8717f6de06,Thank you for understanding. I think very high...,0,0,0,0,0,0
6,00024115d4cbde0f,Please do not add nonsense to Wikipedia. Such ...,0,0,0,0,0,0
7,000247e83dcc1211,:Dear god this site is horrible.,1,0,0,0,0,0
8,00025358d4737918,""" \n Only a fool can believe in such numbers. ...",0,0,0,0,0,0
9,00026d1092fe71cc,== Double Redirects == \n\n When fixing double...,0,0,0,0,0,0


In [ ]:
print("Shape:", final_df.shape)

print("\nMissing values:")
print(final_df.isnull().sum())

print("\nUnique values in prediction columns:")

for label in LABELS:
    print(
        label,
        final_df[label].unique()
    )

print("\nPositive predictions per category:")

for label in LABELS:
    print(
        f"{label:15}: "
        f"{final_df[label].sum():,}"
    )

Shape: (153164, 8)

Missing values:
id               0
comment_text     0
toxic            0
severe_toxic     0
obscene          0
threat           0
insult           0
identity_hate    0
dtype: int64

Unique values in prediction columns:
toxic [1 0]
severe_toxic [0 1]
obscene [1 0]
threat [0 1]
insult [1 0]
identity_hate [1 0]

Positive predictions per category:
toxic          : 38,982
severe_toxic   : 1,762
obscene        : 20,118
threat         : 702
insult         : 18,586
identity_hate  : 2,931


In [ ]:
OUTPUT_PATH = (
    "/content/drive/MyDrive/Data_Science/"
    "test_distilbert_predictions.csv"
)

final_df.to_csv(
    OUTPUT_PATH,
    index=False,
)

print("Saved successfully:")
print(OUTPUT_PATH)

Saved successfully:
/content/drive/MyDrive/Data_Science/test_distilbert_predictions.csv


In [ ]:
from google.colab import files

files.download(OUTPUT_PATH)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>